In [ ]:
import tensorflow as tf


In [ ]:

 
TRAIN_DATASET_DIR = "../data/cat_dog/train"
TEST_DATASET_DIR = "../data/cat_dog/test"
VALIDATION_DATASET_DIR = "../data/cat_dog/validation"

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32

 
train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DATASET_DIR,
    
    # resize every image
    image_size=IMAGE_SIZE,
    
    # how many images per batch
    batch_size=BATCH_SIZE,
    
    # shuffle dataset randomly
    shuffle=True,
    
    # reproducible randomization
    seed=42
)


test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_DATASET_DIR,
    
    # resize every image
    image_size=IMAGE_SIZE,
    
    # how many images per batch
    batch_size=BATCH_SIZE,
    
    # shuffle dataset randomly
    shuffle=True,
    
    # reproducible randomization
    seed=42
)


validation_dataset = tf.keras.utils.image_dataset_from_directory(
    VALIDATION_DATASET_DIR,
    
    # resize every image
    image_size=IMAGE_SIZE,
    
    # how many images per batch
    batch_size=BATCH_SIZE,
    
    # shuffle dataset randomly
    shuffle=True,
    
    # reproducible randomization
    seed=42
)


In [ ]:
print(f'train_dataset: {train_dataset}')
print(f'class names: {train_dataset.class_names}')

## Display a few objects `Dogs and Cow` randomely

In [ ]:
import matplotlib.pyplot as plt

# take one batch
for images, labels in test_dataset.take(1):

    plt.figure(figsize=(10, 10))

    for i in range(9):

        ax = plt.subplot(3, 3, i + 1)

        # image
        plt.imshow(images[i].numpy().astype("uint8"))

        # label
        plt.title(train_dataset.class_names[labels[i]])

        plt.axis("off")

    plt.show()

In [ ]:
for images, labels in  train_dataset.take(1):
    print(images[0].numpy())

## After normalization , byte range will relay on range ```0 to 1```

In [ ]:
 
AUTOTUNE = tf.data.AUTOTUNE

# normalize pixel values
normalization_layer = tf.keras.layers.Rescaling(1./255)

# apply normalization
train_dataset_normalize = train_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

test_dataset_normalize = test_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

validation_dataset_normalize = validation_dataset.map(
    lambda x, y: (normalization_layer(x), y)
)

# optimize pipeline
train_dataset_normalize = train_dataset_normalize.shuffle(1000).prefetch(AUTOTUNE)
test_dataset_normalize = test_dataset_normalize.shuffle(1000).prefetch(AUTOTUNE)
validation_dataset_normalize = validation_dataset_normalize.shuffle(1000).prefetch(AUTOTUNE)

for images, labels in  train_dataset_normalize.take(1):
    print(images[0].numpy())
    # print(images[0].numpy().max())

## validating normalization to denormalization and checking data is truly normalize and we can revert back to actual formal
### after denomalize `Original forms` we will display a few objects

In [ ]:
normalization_layer = tf.keras.layers.Rescaling(255)
train_dataset_denormalize = train_dataset_normalize.map(
    lambda x, y: (normalization_layer(x), y)
)
train_dataset_denormalize = train_dataset_denormalize.shuffle(1000).prefetch(AUTOTUNE)

for images, labels in train_dataset_denormalize.take(1):

    plt.figure(figsize=(10, 10))

    for i in range(9):

        ax = plt.subplot(3, 3, i + 1)

        # image
        plt.imshow(images[i].numpy().astype("uint8"))

        # label
        plt.title(train_dataset.class_names[labels[i]])

        plt.axis("off")

    plt.show()

In [ ]:
conv_2d= tf.keras.layers.Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        input_shape=(224, 224, 3)
    )
# model = tf.keras.Sequential([conv_2d])

# image_batch = tf.expand_dims(image, axis=0)

for images, labels in train_dataset.take(1):
    feature_map = conv_2d(images)
    image_batch = tf.expand_dims(images, axis=0)
    # (batch_size, height, width, channels)
    # batch_size = number of images in the batch
    # height = height of the feature map
    # width = width of the feature map
    # channels = number of filters in the convolutional layer
    print(feature_map.shape)
    # (extra dimension for batch size, height, width, RGB channels)
    # the extra dimension is added because the convolutional layer expects a batch of images as input
    # the batch size is 1 because we are only passing one image at a time
    # the height and width of the feature map are determined by the kernel size and the input image size
    # the number of channels in the feature map is determined by the number of filters in the convolutional layer
    # the input image size is (224, 224, 3) and the kernel size is (3, 3), so the height and width of the feature map will be (222, 222)
    print(image_batch.shape)

In [ ]:
pool_layer = tf.keras.layers.MaxPooling2D(
    pool_size=(2,2)
)
for images, labels in train_dataset_normalize.take(1):
    feature_map = conv_2d(images)
    pooled_feature_map = pool_layer(feature_map)
    print(pooled_feature_map.shape)
    print(feature_map.shape)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation="relu", input_shape=(224,224,3)),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(64, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Conv2D(128, (3,3), activation="relu"),
    tf.keras.layers.MaxPooling2D(2,2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])


model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    train_dataset_normalize,
    validation_data=validation_dataset_normalize,
    epochs=10
)

test_loss, test_acc = model.evaluate(test_dataset_normalize)
model.save("cat_dog_classifier.keras")
print("Test Accuracy:", test_acc)